# Advanced Kubeflow Pipelines: Optimization and Scaling

### What you will learn
- How to configure resource requests and limits (CPU, memory, GPU) for KFP components.
- Auto-scaling pipeline components for efficient resource utilization.
- Managing caching and artifacts to optimize pipeline execution.

### Why this matters in MLOps
ML pipelines can be resource-intensive. Without proper optimization, costs spiral and execution times balloon. Configuring resource limits, enabling intelligent caching, and scaling parallel workers appropriately are essential skills for running efficient production ML workloads.

### You're done when...
- Pipeline components have configured resource requests and limits.
- Caching is configured and understood.
- Parallel execution with resource constraints is implemented.

## The MLOps Perspective
Efficient resource management is the difference between a pipeline that costs $100/run and one that costs $10/run. In cloud/ Kubernetes environments, every request for CPU, memory, or GPU translates directly to cost. Optimization — through caching, right-sizing resources, and parallel execution strategies — is a core MLOps competency that directly impacts the bottom line.

## Setup and Imports

In [ ]:
# Install requirements
!pip install -r requirements.txt

In [ ]:
# Import required libraries
import kfp
from kfp import dsl
from kfp.dsl import component
from kfp import kubernetes as kfp_k8s
from kubernetes import client as k8s_client

## Configuring Resource Requests and Limits
KFP components run as Kubernetes pods. You can specify resource requests (guaranteed minimum) and limits (maximum allowed) for CPU, memory, and GPU. This ensures fair scheduling and prevents runaway resource consumption.

In [ ]:
# Define components with resource configurations
@component(
    base_image="python:3.11",
    packages_to_install=["numpy", "pandas"]
)
def cpu_intensive_preprocessing(data_path: str) -> str:
    """Perform CPU-intensive data preprocessing."""
    import pandas as pd
    import numpy as np
    import time
    df = pd.DataFrame(np.random.randn(10000, 50), columns=[f"col_{i}" for i in range(50)])
    processed_path = "/tmp/processed_data.parquet"
    df.to_parquet(processed_path)
    print(f"Preprocessing complete: shape={df.shape}")
    return processed_path


@component(
    base_image="python:3.11",
    packages_to_install=["torch"]
)
def gpu_training_component(data_path: str, epochs: int) -> str:
    """Simulate GPU-accelerated model training."""
    import torch
    if torch.cuda.is_available():
        print(f"GPU available: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU not available, using CPU")
    model_uri = f"models/gpu_trained_{epochs}epochs"
    print(f"Training for {epochs} epochs completed, model at {model_uri}")
    return model_uri

In [ ]:
# Define a pipeline with resource requests and limits
@dsl.pipeline(
    name="resource-optimized-pipeline",
    description="Pipeline demonstrating resource configuration for components"
)
def resource_optimized_pipeline(data_path: str = "/data/raw", epochs: int = 10):
    preprocess_task = cpu_intensive_preprocessing(data_path=data_path)
    preprocess_task.set_cpu_request("2")
    preprocess_task.set_cpu_limit("4")
    preprocess_task.set_memory_request("4Gi")
    preprocess_task.set_memory_limit("8Gi")

    train_task = gpu_training_component(
        data_path=preprocess_task.output,
        epochs=epochs
    )
    train_task.set_cpu_request("4")
    train_task.set_cpu_limit("8")
    train_task.set_memory_request("16Gi")
    train_task.set_memory_limit("32Gi")
    train_task.set_accelerator_type("nvidia.com/gpu")
    train_task.set_accelerator_limit(1)

## Caching and Artifact Management
KFP caches component outputs by default. When a component is re-run with the same inputs and code, KFP reuses cached results instead of re-executing. This can dramatically reduce execution time and cost.

In [ ]:
# Demonstrate caching behavior
@component(
    base_image="python:3.11"
)
def expensive_computation(param_a: int, param_b: int) -> dict:
    """An expensive computation that benefits from caching."""
    import time
    start = time.time()
    result = sum(i * j for i in range(param_a) for j in range(param_b))
    duration = time.time() - start
    print(f"Computation took {duration:.2f}s, result={result}")
    return {"result": result, "duration": duration}


@dsl.pipeline(
    name="caching-demo-pipeline",
    description="Pipeline demonstrating caching behavior"
)
def caching_demo_pipeline():
    # This component will use cached output on subsequent runs
    comp1 = expensive_computation(param_a=1000, param_b=1000)
    comp1.set_caching_options(enable_caching=True)

    # This component will always re-execute (no caching)
    comp2 = expensive_computation(param_a=500, param_b=500)
    comp2.set_caching_options(enable_caching=False)

## Auto-Scaling Parallel Execution
When running parallel tasks, you can control the degree of parallelism and set resource constraints for each parallel branch. This prevents resource contention and ensures efficient cluster utilization.

In [ ]:
# Define a component for parallel data processing
@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def parallel_worker(worker_id: int, data_chunk: str, complexity: float) -> dict:
    """A worker that processes a data chunk in parallel."""
    import numpy as np
    import time
    delay = complexity * np.random.uniform(0.5, 2.0)
    time.sleep(delay)
    result = {
        "worker_id": worker_id,
        "chunk": data_chunk,
        "processed_records": int(np.random.randint(5000, 50000)),
        "processing_time": delay
    }
    print(f"Worker {worker_id} finished: {result}")
    return result


@component(
    base_image="python:3.11"
)
def merge_parallel_results(results: list) -> dict:
    """Merge results from all parallel workers."""
    total_records = sum(r.get("processed_records", 0) for r in results)
    avg_time = sum(r.get("processing_time", 0) for r in results) / len(results)
    merged = {
        "total_records": total_records,
        "avg_processing_time": avg_time,
        "num_workers": len(results)
    }
    print(f"Merged result: {merged}")
    return merged

In [ ]:
# Pipeline with parallel execution and resource constraints
@dsl.pipeline(
    name="scaled-parallel-pipeline",
    description="Pipeline with auto-scaled parallel execution and resource limits"
)
def scaled_parallel_pipeline(num_workers: int = 4, complexity: float = 1.0):
    worker_tasks = []
    for i in range(num_workers):
        worker = parallel_worker(
            worker_id=i,
            data_chunk=f"/data/chunk_{i}",
            complexity=complexity
        )
        worker.set_cpu_request("500m")
        worker.set_cpu_limit("1")
        worker.set_memory_request("512Mi")
        worker.set_memory_limit("1Gi")
        worker_tasks.append(worker)

    merge_parallel_results(results=worker_tasks)

## GPU Resource Management
GPU resources are expensive and should be allocated carefully. Only request GPUs for components that genuinely need them, and ensure GPU-enabled base images are used.

In [ ]:
# GPU-aware pipeline with conditional GPU allocation
@component(
    base_image="python:3.11",
    packages_to_install=["torch"]
)
def train_small_model(epochs: int) -> str:
    """Train a small model on CPU."""
    import torch
    model = torch.nn.Linear(10, 2)
    model_path = "/tmp/models/small_model.pt"
    torch.save(model.state_dict(), model_path)
    print(f"Small model trained on CPU, saved to {model_path}")
    return model_path


@component(
    base_image="python:3.11",
    packages_to_install=["torch"]
)
def train_large_model(epochs: int, batch_size: int) -> str:
    """Train a large model that benefits from GPU."""
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = torch.nn.Sequential(
        torch.nn.Linear(1000, 512),
        torch.nn.ReLU(),
        torch.nn.Linear(512, 256),
        torch.nn.ReLU(),
        torch.nn.Linear(256, 10)
    ).to(device)
    model_path = f"/tmp/models/large_model_{epochs}e_{batch_size}bs.pt"
    torch.save(model.state_dict(), model_path)
    print(f"Large model trained on {device}, saved to {model_path}")
    return model_path


@dsl.pipeline(
    name="gpu-optimized-pipeline",
    description="Pipeline demonstrating selective GPU allocation"
)
def gpu_optimized_pipeline(epochs: int = 50, batch_size: int = 64):
    # CPU-only component
    small = train_small_model(epochs=10)
    small.set_cpu_request("1").set_cpu_limit("2").set_memory_request("2Gi")

    # GPU-accelerated component
    large = train_large_model(epochs=epochs, batch_size=batch_size)
    large.set_cpu_request("4").set_cpu_limit("8")
    large.set_memory_request("16Gi").set_memory_limit("32Gi")
    large.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1)

## Artifact Management
KFP tracks artifacts produced by pipeline components. Proper artifact management enables lineage tracking, artifact comparison, and efficient storage.

In [ ]:
# Define a pipeline with proper artifact outputs
from kfp.dsl import Dataset, Model, Metrics, Output

@component(
    base_image="python:3.11",
    packages_to_install=["pandas"]
)
def create_artifact_pipeline(
    dataset: Output[Dataset],
    model: Output[Model],
    metrics: Output[Metrics]
) -> None:
    """Create pipeline artifacts with proper KFP typing."""
    import pandas as pd
    import json

    df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
    df.to_csv(dataset.path, index=False)
    dataset.metadata["shape"] = str(df.shape)

    with open(model.path, "w") as f:
        f.write(json.dumps({"model_type": "linear", "coefficients": [0.5, 1.2]}))
    model.metadata["framework"] = "scikit-learn"

    metrics.log_metric("accuracy", 0.95)
    metrics.log_metric("f1_score", 0.92)
    print("Artifacts created successfully")


@dsl.pipeline(
    name="artifact-management-pipeline",
    description="Pipeline demonstrating proper artifact typing and management"
)
def artifact_management_pipeline():
    create_artifact_pipeline()

## Fill-in-the-Blank Exercises

In [ ]:
# Exercise 1: Configure resource limits for a memory-intensive component
@component(base_image="python:3.11")
def memory_intensive_task(size_mb: int) -> str:
    """Task that processes a large array."""
    import numpy as np
    arr = np.random.randn(size_mb * 125000 // 8)  # Approximate memory usage
    return f"Processed {size_mb}MB of data"


@dsl.pipeline(name="exercise-resources")
def exercise_resource_pipeline():
    task = memory_intensive_task(size_mb=1024)
    task.set_memory_request("___")
    task.set_memory_limit("___")
    task.set_cpu_request("___")
    task.set_cpu_limit("___")
# Hint: For 1024MB input, request 2Gi and limit 4Gi; CPU request 1, limit 2

In [ ]:
# Exercise 2: Complete the caching configuration
@dsl.pipeline(name="exercise-caching")
def exercise_caching_pipeline():
    comp = expensive_computation(param_a=100, param_b=200)
    comp.___(enable_caching=___)  # Enable caching

    comp_no_cache = expensive_computation(param_a=50, param_b=100)
    comp_no_cache.___(enable_caching=___)  # Disable caching
# Hint: Use the method set_caching_options with True/False

## Summary
In this notebook, you learned:
- Configuring CPU, memory, and GPU resource requests/limits for components
- Managing caching behavior to optimize pipeline execution
- Implementing auto-scaled parallel execution with resource constraints
- Managing pipeline artifacts with KFP's type system
- These optimization techniques reduce costs and improve pipeline efficiency